This notebook is adapted from https://github.com/ZhaofengWu/counterfactual-evaluation/tree/master/arithmetic 

In [ ]:
import json
import numpy as np
import random

In [ ]:
def templatize(expr, base, cot=True, n_shots=0):
    digits = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    if cot:
        return f"You are a mathematician. Assuming that all numbers are in base-{base} where the digits are \"{digits[:base]}\", what is {expr}? Let's think step by step, and end the response with the result in \"\\boxed{{result}}\"."
    else:
        return f"You are a mathematician. Assuming that all numbers are in base-{base} where the digits are \"{digits[:base]}\", what is {expr}? End the response with the result in \"\\boxed{{result}}\"."

def templatize_trivial(expr):
    return f"What is {expr}? End the response with the result in \"\\boxed{{result}}\"."

def add_in_base(num1, num2, base):
    """
    Adds two positive integers in a given base.

    Args:
        num1 (str): First number as a string.
        num2 (str): Second number as a string.
        base (int): Base of the numbers (2 <= base <= 36).

    Returns:
        str: Sum of num1 and num2 in the same base.
    """
    lhs_base10 = int(num1, base)
    rhs_base10 = int(num2, base)
    sum_base10 = lhs_base10 + rhs_base10
    return np.base_repr(sum_base10, base)

def sample_number(n_digits, base):
    digits = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"[:base]
    number = "".join(str(random.choice(digits[1:] if i == 0 else digits)) for i in range(n_digits))
    assert number[0] != "0" and len(number) == n_digits  # i'm paranoid
    return number

def expr_is_hard(left, right, base):
    if any("A" <= d <= "Z" for d in left + right):
        return True
    label = add_in_base(left, right, base)
    base10_label = str(int(left) + int(right))
    return label != base10_label

def sample_single(n_digits_1, base, n_digits_2=None):
    if n_digits_2 is None:
        n_digits_2 = n_digits_1
    left = sample_number(n_digits_1, base)
    right = sample_number(n_digits_2, base)
    if base != 10:
        while not expr_is_hard(left, right, base):
            left = sample_number(n_digits, base)
            right = sample_number(n_digits, base)
    label = add_in_base(left, right, base)
    return left, right, label


In [3]:
# Example usage
num1 = "16"
num2 = "37"
base = 9

sum_in_base = add_in_base(num1, num2, base)
print(f"The sum of {num1} and {num2} in base {base} is: {sum_in_base}")

The sum of 16 and 37 in base 9 is: 54


In [4]:
templatize(f"{num1} + {num2}", base)

'You are a mathematician. Assuming that all numbers are in base-9 where the digits are "012345678", what is 16 + 37? Let\'s think step by step, and end the response with the result in "\\boxed{result}".'

In [ ]:
num_samples = 1000
pairs = []
while len(pairs) < num_samples:
    base = random.choice([7,9,11,12])
    p, q, label = sample_single(2, base)
    if (p, q) not in pairs and (q, p) not in pairs:
        pairs.append((p, q, base, label))
random.shuffle(pairs) # The rejection create non-uniform order
data = [{
    "id": i,
    "Question": templatize(f"{p} + {q}", base),
    "answer": label,
    "subset": "counterfact_arith",
    "p_q": (p, q),
}  for i, (p, q, base, label) in enumerate(pairs)]

In [7]:
with open("arithmetic_trivial.json", "w") as f:
    json.dump(data, f)